# Annotation Comparison

Visualizes annotated instances from the event sequencing task. For each instance, the text is shown with the two pre-highlighted spans, followed by each annotator's response (is each span an event? if both are events: sequencing and causality ratings).

In [1]:
import json
import csv
import ast
from IPython.display import display, HTML
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

## Load Data

In [ ]:
CSV_PATH = '../data/dolma_combined_final_sample_700_with_llm_summary_safeid_with_spans.csv'
RESULTS_DIR = '../annotation_output/results'
# ANNOTATORS = ['adde1214', 'tejo9855', 'umasgunturi@gmail.com']
ANNOTATORS = ['adde1214', 'tejo9855']


# ── Load text data ──────────────────────────────────────────────────────────
csv.field_size_limit(10**7)
with open(CSV_PATH) as f:
    reader = csv.DictReader(f)
    rows_in_order = list(reader)

text_rows = {row['safe_instance_id']: row for row in rows_in_order}
dataset_order = [row['safe_instance_id'] for row in rows_in_order]

print(f'Loaded {len(text_rows)} instances from CSV')

# ── Load annotation data ─────────────────────────────────────────────────────
def parse_annotation(val):
    """Extract annotation dict from instance_id_to_label_to_value entry."""
    inner = val[0][1]
    if not inner:
        return None
    parsed = json.loads(inner)
    return parsed[0] if parsed else None

annotator_data = {}  # annotator -> {inst_id -> annotation_dict}
for ann in ANNOTATORS:
    with open(f'{RESULTS_DIR}/{ann}/user_state.json') as f:
        state = json.load(f)
    annotator_data[ann] = {
        inst_id: parse_annotation(val)
        for inst_id, val in state['instance_id_to_label_to_value'].items()
    }

for ann, d in annotator_data.items():
    print(f'{ann}: {len(d)} instances annotated')

Loaded 679 instances from CSV
adde1214_likert: 41 instances annotated
tejo9855_likert: 51 instances annotated


## Helper Functions

In [7]:
SPAN1_COLOR = '#4e9af1'   # blue
SPAN2_COLOR = '#f4a432'   # orange
SPAN1_LABEL = 'Span 1'
SPAN2_LABEL = 'Span 2'

def render_text_with_spans(text, span1, span2):
    """Return HTML string with the two spans highlighted in the passage."""
    s1_start, s1_end = span1['start'], span1['end']
    s2_start, s2_end = span2['start'], span2['end']

    # Sort spans by position so we can insert tags left-to-right
    spans = sorted(
        [('span1', s1_start, s1_end), ('span2', s2_start, s2_end)],
        key=lambda x: x[1]
    )

    colors = {'span1': SPAN1_COLOR, 'span2': SPAN2_COLOR}
    labels = {'span1': SPAN1_LABEL, 'span2': SPAN2_LABEL}

    result = []
    cursor = 0
    for name, start, end in spans:
        result.append(text[cursor:start])
        color = colors[name]
        label = labels[name]
        token = text[start:end]
        result.append(
            f'<mark style="background:{color};color:#fff;padding:2px 5px;'
            f'border-radius:3px;font-weight:bold;" '
            f'title="{label}">{token}</mark>'
        )
        cursor = end
    result.append(text[cursor:])

    body = ''.join(result)
    body = body.replace('\n', '<br>')
    return body


def annotation_summary_html(ann_name, ann):
    """Return an HTML row summarising one annotator's response."""
    if ann is None:
        return f'<tr><td><b>{ann_name}</b></td><td colspan="4" style="color:#999">— not annotated —</td></tr>'

    s1 = '✅ Event' if ann.get('span1_is_event') else '❌ Not event'
    s2 = '✅ Event' if ann.get('span2_is_event') else '❌ Not event'

    if ann.get('span1_is_event') and ann.get('span2_is_event'):
        seq = ann.get('sequencing_rating', '—')
        caus = ann.get('causality_rating', '—')
    else:
        seq = caus = '—'

    return (
        f'<tr>'
        f'<td style="padding:4px 10px"><b>{ann_name}</b></td>'
        f'<td style="padding:4px 14px;color:{SPAN1_COLOR}">{s1}</td>'
        f'<td style="padding:4px 14px;color:{SPAN2_COLOR}">{s2}</td>'
        f'<td style="padding:4px 14px">{seq}</td>'
        f'<td style="padding:4px 14px">{caus}</td>'
        f'</tr>'
    )


def display_instance(inst_id, min_annotators=1):
    """Render one instance as rich HTML."""
    row = text_rows.get(inst_id)
    if row is None:
        print(f'No text found for {inst_id}')
        return

    # Collect annotators who have this instance
    anns = {ann: annotator_data[ann].get(inst_id) for ann in ANNOTATORS
            if inst_id in annotator_data[ann]}
    if len(anns) < min_annotators:
        return

    # Grab span info from any completed annotation (all instances share the same spans)
    sample_ann = next((a for a in anns.values() if a is not None), None)
    if sample_ann is None:
        return  # No completed annotation to pull span positions from

    span1 = sample_ann['span1']
    span2 = sample_ann['span2']

    text = row['sampled_text']
    highlighted = render_text_with_spans(text, span1, span2)

    # Legend
    legend = (
        f'<span style="background:{SPAN1_COLOR};color:#fff;padding:2px 7px;border-radius:3px;margin-right:6px">'
        f'{SPAN1_LABEL}: <b>{span1["token"]}</b></span>'
        f'<span style="background:{SPAN2_COLOR};color:#fff;padding:2px 7px;border-radius:3px">'
        f'{SPAN2_LABEL}: <b>{span2["token"]}</b></span>'
    )

    # Annotator table
    rows_html = ''.join(annotation_summary_html(ann, data) for ann, data in anns.items())
    table = (
        '<table style="border-collapse:collapse;margin-top:8px;font-size:0.92em">'
        '<thead><tr style="background:#f0f0f0">'
        '<th style="padding:4px 10px;text-align:left">Annotator</th>'
        f'<th style="padding:4px 14px;text-align:left;color:{SPAN1_COLOR}">Span 1 ({span1["token"]})</th>'
        f'<th style="padding:4px 14px;text-align:left;color:{SPAN2_COLOR}">Span 2 ({span2["token"]})</th>'
        '<th style="padding:4px 14px;text-align:left">Sequencing (1–5)</th>'
        '<th style="padding:4px 14px;text-align:left">Causality (1–5)</th>'
        '</tr></thead>'
        f'<tbody>{rows_html}</tbody>'
        '</table>'
    )

    source = row.get('source', '')
    doc_id = row.get('id', inst_id)

    html = (
        '<div style="border:1px solid #ddd;border-radius:6px;padding:14px 18px;margin-bottom:20px;font-family:sans-serif">'
        f'<div style="font-size:0.8em;color:#888;margin-bottom:6px">{inst_id} &nbsp;|&nbsp; {source} &nbsp;|&nbsp; {doc_id}</div>'
        f'<div style="margin-bottom:8px">{legend}</div>'
        f'<div style="background:#fafafa;border-left:3px solid #ccc;padding:10px 14px;font-size:0.93em;line-height:1.6;max-height:260px;overflow-y:auto">'
        f'{highlighted}</div>'
        f'{table}'
        '</div>'
    )
    display(HTML(html))

## Agreement Metrics

Computed over instances annotated by **all annotators**.

- **Event label** (span1 / span2): binary exact match + Cohen's κ  
- **Sequencing / Causality ratings**: restricted to instances where both annotators marked both spans as events; exact match + linear-weighted Cohen's κ (appropriate for ordinal Likert scales)

In [8]:
all_inst_sets = [set(annotator_data[ann].keys()) for ann in ANNOTATORS]
shared_all = set.intersection(*all_inst_sets)
print(f'{len(shared_all)} instances annotated by all annotators')

41 instances annotated by all annotators


In [9]:
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr

ANN1, ANN2 = ANNOTATORS[0], ANNOTATORS[1]

# ── Collect paired annotations over shared instances (dataset order) ─────────
span1_event, span2_event = [], []   # binary lists per annotator pair
seq_ratings, caus_ratings = [], []  # Likert lists (only when both rated)

for inst_id in dataset_order:
    if inst_id not in shared_all:
        continue
    a1 = annotator_data[ANN1].get(inst_id)
    a2 = annotator_data[ANN2].get(inst_id)
    if a1 is None or a2 is None:
        continue

    span1_event.append((bool(a1.get('span1_is_event')), bool(a2.get('span1_is_event'))))
    span2_event.append((bool(a1.get('span2_is_event')), bool(a2.get('span2_is_event'))))

    both_events_a1 = a1.get('span1_is_event') and a1.get('span2_is_event')
    both_events_a2 = a2.get('span1_is_event') and a2.get('span2_is_event')
    if both_events_a1 and both_events_a2:
        s1, s2 = a1.get('sequencing_rating'), a2.get('sequencing_rating')
        c1, c2 = a1.get('causality_rating'),  a2.get('causality_rating')
        if s1 is not None and s2 is not None:
            seq_ratings.append((int(s1), int(s2)))
        if c1 is not None and c2 is not None:
            caus_ratings.append((int(c1), int(c2)))


# ── Metric helpers ────────────────────────────────────────────────────────────
def exact_match(pairs):
    if not pairs: return None
    return sum(a == b for a, b in pairs) / len(pairs)

def within_one(pairs):
    if not pairs: return None
    return sum(abs(a - b) <= 1 for a, b in pairs) / len(pairs)

def mae(pairs):
    if not pairs: return None
    return np.mean([abs(a - b) for a, b in pairs])

def kappa(pairs, weights=None):
    if len(pairs) < 2: return None
    y1, y2 = zip(*pairs)
    if len(set(y1) | set(y2)) < 2: return None
    return cohen_kappa_score(y1, y2, weights=weights)

def icc_two_way_random(pairs):
    """ICC(2,1) — two-way random effects, single measures.
    Explicitly models per-rater severity bias; agreement is credited even
    when one annotator is systematically higher/lower than the other."""
    if len(pairs) < 2: return None
    data = np.array(pairs, dtype=float)   # shape (n, 2)
    n, k = data.shape
    grand_mean    = data.mean()
    subject_means = data.mean(axis=1)
    rater_means   = data.mean(axis=0)
    SS_total    = ((data - grand_mean) ** 2).sum()
    SS_subjects = k * ((subject_means - grand_mean) ** 2).sum()
    SS_raters   = n * ((rater_means   - grand_mean) ** 2).sum()
    SS_error    = SS_total - SS_subjects - SS_raters
    MS_subjects = SS_subjects / (n - 1)
    MS_raters   = SS_raters   / (k - 1)
    MS_error    = SS_error    / ((n - 1) * (k - 1))
    denom = MS_subjects + (k - 1) * MS_error + k * (MS_raters - MS_error) / n
    if denom == 0: return None
    return (MS_subjects - MS_error) / denom

def annotator_means(pairs):
    if not pairs: return None, None
    y1, y2 = zip(*pairs)
    return np.mean(y1), np.mean(y2)


# ── Results table ─────────────────────────────────────────────────────────────
def fmt(v, pct=False):
    if v is None: return '<span style="color:#aaa">—</span>'
    return f'{v*100:.1f}%' if pct else f'{v:.3f}'

# (label, pairs, kappa_weights, include_ordinal_metrics)
metrics = [
    ('Span 1 is event',  span1_event,  None,     False),
    ('Span 2 is event',  span2_event,  None,     False),
    ('Sequencing (1–5)', seq_ratings,  'linear', True),
    ('Causality (1–5)',  caus_ratings, 'linear', True),
]

rows_html = ''
for label, pairs, w, ordinal in metrics:
    n    = len(pairs)
    em   = exact_match(pairs)
    w1   = within_one(pairs) if ordinal else None
    m_ae = mae(pairs)        if ordinal else None
    k    = kappa(pairs, weights=w)
    icc  = icc_two_way_random(pairs) if ordinal else None
    m1, m2 = annotator_means(pairs) if ordinal else (None, None)

    mean_str = (
        f'{ANN1}: {m1:.2f} &nbsp; {ANN2}: {m2:.2f}'
        if m1 is not None else '<span style="color:#aaa">—</span>'
    )

    rows_html += (
        f'<tr>'
        f'<td style="padding:6px 14px">{label}</td>'
        f'<td style="padding:6px 14px;text-align:center">{n}</td>'
        f'<td style="padding:6px 14px;text-align:center">{fmt(em, pct=True)}</td>'
        f'<td style="padding:6px 14px;text-align:center">{fmt(w1, pct=True)}</td>'
        f'<td style="padding:6px 14px;text-align:center">{fmt(m_ae)}</td>'
        f'<td style="padding:6px 14px;text-align:center">{fmt(k)}</td>'
        f'<td style="padding:6px 14px;text-align:center">{fmt(icc)}</td>'
        f'<td style="padding:6px 14px;font-size:0.85em;color:#555">{mean_str}</td>'
        f'</tr>'
    )

header = (
    '<tr style="background:#f0f0f0">'
    '<th style="padding:6px 14px;text-align:left">Dimension</th>'
    '<th style="padding:6px 14px">N</th>'
    '<th style="padding:6px 14px">Exact match</th>'
    '<th style="padding:6px 14px">Within 1</th>'
    '<th style="padding:6px 14px">MAE</th>'
    '<th style="padding:6px 14px">Cohen\'s κ</th>'
    '<th style="padding:6px 14px">ICC(2,1)</th>'
    '<th style="padding:6px 14px;text-align:left">Mean rating per annotator</th>'
    '</tr>'
)

notes = (
    '<p style="color:#777;font-size:0.8em;margin-top:10px;line-height:1.5">'
    '<b>Within 1</b>: fraction of pairs where ratings differ by ≤1 point — robust to small N and ceiling effects.<br>'
    '<b>MAE</b>: mean absolute difference in ratings — directly interpretable scale of disagreement.<br>'
    '<b>Cohen\'s κ</b>: linear-weighted for Likert; penalises large disagreements more.<br>'
    '<b>ICC(2,1)</b>: two-way random effects model; explicitly partitions rater severity bias from true disagreement.<br>'
    'Sequencing/Causality N may be lower if either annotator marked a span as non-event.'
    '</p>'
)

table_html = (
    f'<div style="font-family:sans-serif;margin-top:12px">'
    f'<b>{ANN1}</b> vs <b>{ANN2}</b> &nbsp;|&nbsp; {len(span1_event)} shared instances'
    f'<table style="border-collapse:collapse;margin-top:10px;font-size:0.93em">'
    f'<thead>{header}</thead>'
    f'<tbody>{rows_html}</tbody>'
    f'</table>'
    f'{notes}'
    f'</div>'
)
display(HTML(table_html))

Dimension,N,Exact match,Within 1,MAE,Cohen's κ,"ICC(2,1)",Mean rating per annotator
Span 1 is event,41,75.6%,—,—,0.345,—,—
Span 2 is event,41,73.2%,—,—,0.199,—,—
Sequencing (1–5),19,31.6%,78.9%,0.895,-0.073,-0.009,adde1214_likert: 4.42 tejo9855_likert: 4.26
Causality (1–5),19,26.3%,52.6%,1.421,0.207,0.297,adde1214_likert: 3.47 tejo9855_likert: 2.26


In [10]:
print(f"seq_ratings  (n={len(seq_ratings)}): {seq_ratings}")
print(f"caus_ratings (n={len(caus_ratings)}): {caus_ratings}")

if seq_ratings:
    y1, y2 = zip(*seq_ratings)
    print(f"\nSequencing  — {ANN1}: {list(y1)}")
    print(f"Sequencing  — {ANN2}: {list(y2)}")
if caus_ratings:
    y1, y2 = zip(*caus_ratings)
    print(f"\nCausality   — {ANN1}: {list(y1)}")
    print(f"Causality   — {ANN2}: {list(y2)}")

seq_ratings  (n=19): [(5, 5), (3, 4), (5, 3), (4, 5), (5, 4), (5, 4), (5, 5), (5, 4), (5, 5), (5, 3), (5, 3), (5, 5), (5, 5), (3, 3), (4, 5), (3, 5), (4, 3), (4, 5), (4, 5)]
caus_ratings (n=19): [(4, 4), (4, 2), (4, 2), (1, 2), (4, 3), (1, 1), (5, 2), (5, 3), (4, 3), (5, 1), (4, 1), (4, 4), (3, 1), (1, 1), (3, 3), (1, 2), (3, 1), (5, 4), (5, 3)]

Sequencing  — adde1214_likert: [5, 3, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 4, 3, 4, 4, 4]
Sequencing  — tejo9855_likert: [5, 4, 3, 5, 4, 4, 5, 4, 5, 3, 3, 5, 5, 3, 5, 5, 3, 5, 5]

Causality   — adde1214_likert: [4, 4, 4, 1, 4, 1, 5, 5, 4, 5, 4, 4, 3, 1, 3, 1, 3, 5, 5]
Causality   — tejo9855_likert: [4, 2, 2, 2, 3, 1, 2, 3, 3, 1, 1, 4, 1, 1, 3, 2, 1, 4, 3]


## Instances Annotated by All 4 Annotators

In [ ]:
for inst_id in dataset_order:
    if inst_id in shared_all:
        display_instance(inst_id)

21 instances annotated by all annotators


Annotator,Span 1 (flung),Span 2 (attack),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,4
tejo9855,✅ Event,✅ Event,5,4


Annotator,Span 1 (campaigning),Span 2 (promises),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,3,4
tejo9855,✅ Event,✅ Event,4,2


Annotator,Span 1 (took),Span 2 (assured),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,4
tejo9855,✅ Event,✅ Event,3,2


Annotator,Span 1 (shines),Span 2 (rides),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,4,1
tejo9855,✅ Event,✅ Event,5,2


...

Annotator,Span 1 (named),Span 2 (said),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,5
tejo9855,✅ Event,✅ Event,3,1


Annotator,Span 1 (comments),Span 2 (says),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,—,—
tejo9855,❌ Not event,❌ Not event,—,—


Annotator,Span 1 (approached),Span 2 (said),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,4
tejo9855,✅ Event,✅ Event,3,1


Annotator,Span 1 (met),Span 2 (hosted),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,—,—
tejo9855,✅ Event,✅ Event,—,—


Annotator,Span 1 (considering),Span 2 (remaining),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,❌ Not event,—,—
tejo9855,❌ Not event,❌ Not event,—,—


## All Annotated Instances (grouped by annotator)

In [34]:
for ann in ANNOTATORS:
    ann_ids = set(annotator_data[ann].keys())
    ordered = [inst_id for inst_id in dataset_order if inst_id in ann_ids]
    display(HTML(f'<h3 style="font-family:sans-serif;margin-top:24px">Annotator: {ann} ({len(ordered)} instances)</h3>'))
    for inst_id in ordered:
        display_instance(inst_id)

Annotator,Span 1 (flung),Span 2 (attack),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,4
tejo9855,✅ Event,✅ Event,5,4


Annotator,Span 1 (campaigning),Span 2 (promises),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,3,4
tejo9855,✅ Event,✅ Event,4,2


Annotator,Span 1 (took),Span 2 (assured),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,4
tejo9855,✅ Event,✅ Event,3,2


Annotator,Span 1 (shines),Span 2 (rides),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,4,1
tejo9855,✅ Event,✅ Event,5,2


Annotator,Span 1 (gave),Span 2 (denounce),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,—,—
tejo9855,❌ Not event,❌ Not event,—,—


...

Annotator,Span 1 (comments),Span 2 (says),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,—,—
tejo9855,❌ Not event,❌ Not event,—,—


Annotator,Span 1 (approached),Span 2 (said),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,5,4
tejo9855,✅ Event,✅ Event,3,1


Annotator,Span 1 (met),Span 2 (hosted),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,✅ Event,—,—
tejo9855,✅ Event,✅ Event,—,—


Annotator,Span 1 (considering),Span 2 (remaining),Sequencing (1–5),Causality (1–5)
adde1214,✅ Event,❌ Not event,—,—
tejo9855,❌ Not event,❌ Not event,—,—


Annotator,Span 1 (Drive),Span 2 (created),Sequencing (1–5),Causality (1–5)
tejo9855,❌ Not event,✅ Event,—,—
